In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import onnxruntime as ort
from pathlib import Path
tf.config.set_visible_devices([], 'GPU')  # force CPU

Keys of the model: dict_keys(['_self_setattr_tracking', '_self_unconditional_checkpoint_dependencies', '_self_unconditional_dependency_names', '_self_unconditional_deferred_dependencies', '_self_update_uid', '_self_name_based_restores', '_self_saveable_object_factories', '_tf_var_leaves', '_methods', 'signatures', 'graph_debug_info', 'tensorflow_version', 'tensorflow_git_version'])
Tensor flow version: 2.20.0
Total number of parameters: 101757263


# Load Model  
If onnx model is available use it
If not use perch_v2_cpu model

In [12]:
USE_ONNX = True
ONNX_PATH = Path('../models/perch_onnx/perch_v2.onnx')
if USE_ONNX:
    session_option = ort.SessionOptions()
    session_option.intra_op_num_threads = 4
    onnx_session = ort.InferenceSession(str(ONNX_PATH), sess_options = session_option, \
                                        providers=['CPUExecutionProvider'])
    # get the name of the input
    onnx_ipt_name = onnx_session.get_inputs()[0].name
    print(f'Onnx input name: {onnx_ipt_name}')
    # map the names of onnx outputs to integers
    onnx_opt_map = {o.name: i for i,o in enumerate(onnx_session.get_outputs())}
    print (f'Onnx output maps: {onnx_opt_map}')
else:
    # normal version
    model = tf.saved_model.load('../models/perch_v2_cpu')
    infer = model.signatures["serving_default"]
    print(f'Keys of the model: {model.__dict__.keys()}')
    print(f'Tensor flow version: {model.tensorflow_version}')
    print(f'Total number of parameters: {sum(tf.size(v).numpy() for v in model._tf_var_leaves)}')

Onnx input name: inputs
Onnx output maps: {'embedding': 0, 'spatial_embedding': 1, 'spectrogram': 2, 'label': 3}


101757263


## Load a 5-second .ogg chunk → numpy array at 32 kHz

Perch v2 expects `float32` waveform, shape `(batch, 160000)`, at **32 kHz**.

In [13]:
import librosa
import numpy as np

SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS  # 160000

def load_chunk(path: str, offset_sec: float) -> np.ndarray:
    """Load one 5-second chunk from an .ogg file.
    
    Returns float32 array of shape (160000,), ready for Perch v2.
    """
    waveform, _ = librosa.load(
        path,
        sr=SAMPLE_RATE,
        offset=offset_sec,
        duration=CHUNK_SECONDS,
        mono=True,
    )
    # Pad if the file ends before 5 seconds
    if len(waveform) < CHUNK_SAMPLES:
        waveform = np.pad(waveform, (0, CHUNK_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)

# Quick sanity check on one file
sample_path = '../data/train_audio/22930/iNat317238.ogg'  
chunk = load_chunk(sample_path, offset_sec=0.0)
print('Shape:', chunk.shape)   # (160000,)
print('dtype:', chunk.dtype)   # float32
print('Range:', chunk.min(), chunk.max())

Shape: (160000,)
dtype: float32
Range: -0.45734373 0.5143107


## Extract embeddings with Perch v2

`model2.infer_tf` returns a dict with keys: `embedding` (1536-d), `label` (14795 logits), `spectrogram`, `spatial_embedding`.  
We only need `embedding`.

In [15]:
# infer = model.signatures['serving_default']

def extract_embedding(waveform: np.ndarray) -> np.ndarray:
    """waveform: (160000,) float32  →  embedding: (1536,) float32"""
    inp = waveform[np.newaxis, :]
    if USE_ONNX:
        outs = onnx_session.run(None, {onnx_ipt_name: inp})
        logits = outs[onnx_opt_map['label']].astype(np.float32)
        emb = outs[onnx_opt_map['embedding']].astype(np.float32) 
        
    else:
           # (1, 160000)
        out = infer(inputs=tf.constant(tf.convert_to_tensor(inp)))
        logits = out["label"].numpy().astype(np.float32)
        emb    = out["embedding"].numpy().astype(np.float32)
    return emb[0], logits           # (1536,)

emb, logits = extract_embedding(chunk)
print(f'Embedding shape: {emb.shape}')  # (1536,)
print(f'Label shape: {logits.shape}')

Embedding shape: (1536,)
Label shape: (1, 14795)


## Pre-extract and cache all embeddings then save to .npy

Extract embeddings once, cache to disk, then train the head on raw numpy arrays to accelerate training.

### Chunking the long files

In [17]:
import pandas as pd
from tqdm.notebook import tqdm
from pathlib import Path
import math
import soundfile as sf
files = Path('../data/train_audio').rglob('*.ogg')
train_df = pd.read_csv('../data/train.csv')
taxonomy_df = pd.read_csv('../data/taxonomy.csv')
# taxonomy_set is sorted in ascending order as a baseline for audios in
# train_audio, train_sounscrapes, and test_soundscrapes 
taxonomy_set = sorted(set((taxonomy_df['primary_label'].unique())))
label2idx = {label: idx for idx, label in enumerate(taxonomy_set)}
train_df = train_df[['primary_label', 'filename']] # remove other columns for faster processing
train_df['file_path'] = '../data/train_audio/' + train_df['filename']
FIXED_LENGTH = 5 # the duration of a standard audio in seconds
def get_chunks_number(file_path:str):
    '''
    returns the number of chunks
    for the file in row idx of train_df
    '''
    return math.ceil(sf.info(file_path).duration/FIXED_LENGTH) # math.ceil to make sure 18.024 -> 4 chunnks, 15 -> 3 chunks
# add new rows for the new chunks separated from long files
new_idx = train_df.index.repeat(train_df['file_path'].apply(get_chunks_number)) 
train_df = train_df.loc[new_idx].reset_index(drop=True)
# add new offsets (offseting from the begining of long files) in seconds  
train_df['offset_sec'] = train_df.groupby('file_path').cumcount()*FIXED_LENGTH
# plan to use Efficientnet model, which requires labels are encoded in number
train_df['encoded_label'] = train_df['primary_label'].map(label2idx)
train_df.head(3)

,primary_label,filename,file_path,offset_sec,encoded_label
0,1161364,1161364/iNat1216197.ogg,../data/train_audio/1161364/iNat1216197.ogg,0,0
1,1161364,1161364/iNat1216197.ogg,../data/train_audio/1161364/iNat1216197.ogg,5,0
2,1161364,1161364/iNat1216197.ogg,../data/train_audio/1161364/iNat1216197.ogg,10,0


In [18]:
train_df = train_df[['file_path', 'offset_sec', 'encoded_label']]
train_df.to_parquet('../data/chunks.parquet', index=False)
print(len(train_df))
train_df.head(2)

265924


,file_path,offset_sec,encoded_label
0,../data/train_audio/1161364/iNat1216197.ogg,0,0
1,../data/train_audio/1161364/iNat1216197.ogg,5,0


In [ ]:


# Load pre-built chunks dataframe (same one used for EfficientNet training)
chunks_df = pd.read_parquet('../data/chunks.parquet')   # columns: file_path, offset_sec, encoded_label

EMB_CACHE = Path('../data/perch_embeddings.npy')
LBL_CACHE = Path('../data/perch_labels.npy')

if not EMB_CACHE.exists():
    embeddings, labels = [], []
    for _, row in tqdm(chunks_df.iterrows(), total=len(chunks_df)):
        wav = load_chunk(row['file_path'], row['offset_sec'])
        emb = extract_embedding(wav)
        embeddings.append(emb)
        labels.append(row['encoded_label'])
    embeddings = np.stack(embeddings).astype(np.float32)
    labels     = np.array(labels, dtype=np.int64)
    np.save(EMB_CACHE, embeddings)
    np.save(LBL_CACHE, labels)
    print(f'Saved {len(embeddings)} embeddings → {EMB_CACHE}')
else:
    embeddings = np.load(EMB_CACHE)
    labels     = np.load(LBL_CACHE)
    print(f'Loaded {len(embeddings)} cached embeddings')

print('embeddings:', embeddings.shape)  # (N, 1536)
print('labels:    ', labels.shape)      # (N,)

## Train a lightweight PyTorch head

The head is just `Linear(1536 → 234)` with dropout. Perch weights stay frozen.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

NUM_CLASSES = 234
EMBED_DIM   = 1536
BATCH_SIZE  = 256
NUM_EPOCHS  = 20
LR          = 1e-3

# ---- Dataset from cached numpy arrays ----
X = torch.from_numpy(embeddings)   # (N, 1536) float32
y = torch.from_numpy(labels)       # (N,)      int64

dataset = TensorDataset(X, y)
n_train = int(0.8 * len(dataset))
n_val   = int(0.1 * len(dataset))
n_test  = len(dataset) - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
                                          generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---- Lightweight head ----
class PerchHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(EMBED_DIM),
            nn.Dropout(0.3),
            nn.Linear(EMBED_DIM, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, NUM_CLASSES),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
head = PerchHead().to(device)
print(head)
print(f'Head parameters: {sum(p.numel() for p in head.parameters()):,}')

In [ ]:
import mlflow, mlflow.pytorch

optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

mlflow.set_experiment('birdclef2026-perch-head')

best_val_loss = float('inf')
with mlflow.start_run(run_name='perch_v2_head'):
    mlflow.log_params({'epochs': NUM_EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE,
                       'embed_dim': EMBED_DIM, 'num_classes': NUM_CLASSES})

    for epoch in range(NUM_EPOCHS):
        # --- train ---
        head.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(head(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- validate ---
        head.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = head(xb)
                val_loss += criterion(preds, yb).item()
                correct  += (preds.argmax(1) == yb).sum().item()
                total    += len(yb)

        train_loss /= len(train_loader)
        val_loss   /= len(val_loader)
        val_acc     = correct / total
        scheduler.step()

        mlflow.log_metrics({'train_loss': train_loss, 'val_loss': val_loss,
                            'val_acc': val_acc}, step=epoch)
        print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS}  '
              f'train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            mlflow.pytorch.log_model(head, 'best_model')

print('Training complete. Best val loss:', best_val_loss)